# LegalQA Main Stage 1/3 — QLoRA training

Notebook Kaggle này chỉ chuẩn bị tập SFT, chạy lexical retrieval và fine-tune QLoRA. Nó không chạy dev30, dev100 hay submission, nhờ đó checkpoint có thể hoàn tất trong một phiên 12 giờ.

Output quan trọng là toàn bộ thư mục `legalqa_main_stage1_train_v8/sft`. Sau khi Save & Run All thành công, thêm output notebook này làm Input cho Stage 2 (hoặc xuất nó thành Kaggle Dataset).

## 1. Cấu hình Kaggle

In [ ]:
from pathlib import Path
import hashlib, json, shutil, subprocess, sys

KAGGLE_ROOT = Path('/kaggle')
if not KAGGLE_ROOT.exists():
    raise RuntimeError('Notebook này chỉ chạy trên Kaggle.')

WORK_BASE = Path('/kaggle/working')
REPO_URL = 'https://github.com/lighth-gh/uit-dsc-2026-task2-legalqa.git'
REPO_REF = 'main'
CODE = WORK_BASE / 'uit-dsc-2026-task2-legalqa'

VERSION3_URL = 'https://www.kaggle.com/datasets/lighth/ver3-smoke-output'
VERSION3_ROOT = Path('/kaggle/input/datasets/lighth/ver3-smoke-output/legalqa_smoke_full_v1')
SAVED_INDEX = VERSION3_ROOT / 'index'
SAVED_MODELS = VERSION3_ROOT / 'models'
DATASET_ROOT = Path('/kaggle/input/datasets/lighth/uit-dsc-2026-task2-legalqa-train')
TRAIN_PATH = DATASET_ROOT / 'train.json'
TEST_PATH = DATASET_ROOT / 'public-official.json'

RUN_ROOT = WORK_BASE / 'legalqa_main_stage1_train_v8'
MODELS = RUN_ROOT / 'models'
DATA = RUN_ROOT / 'data_public'
INDEX = SAVED_INDEX
CFG = RUN_ROOT / 'config.json'
SFT_DIR = RUN_ROOT / 'sft'
RESUME_CHECKPOINT = None  # Có thể trỏ tới một checkpoint đầy đủ trong /kaggle/input.

RUN_ROOT.mkdir(parents=True, exist_ok=True)
print('Stage 1 output:', RUN_ROOT)

## 2. Clone code và kiểm tra input

In [ ]:
if CODE.exists():
    if not (CODE / '.git').is_dir():
        raise RuntimeError(f'{CODE} tồn tại nhưng không phải Git repo. Hãy Restart Session.')
    current_remote = subprocess.check_output(['git', '-C', str(CODE), 'remote', 'get-url', 'origin'], text=True).strip()
    if current_remote.rstrip('/') != REPO_URL.rstrip('/'):
        raise RuntimeError(f'Remote không đúng: {current_remote}')
    subprocess.run(['git', '-C', str(CODE), 'pull', '--ff-only', 'origin', REPO_REF], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(CODE)], check=True)

for label, path in [('train', TRAIN_PATH), ('test', TEST_PATH)]:
    if not path.is_file():
        raise FileNotFoundError(f'Thiếu {label}: {path}')
required_index = ['index_manifest.json', 'corpus.sqlite', 'dense.faiss']
missing_index = [name for name in required_index if not (SAVED_INDEX / name).is_file()]
missing_models = [role for role in ['embedding', 'reranker', 'generator'] if not (SAVED_MODELS / role / 'config.json').is_file()]
if missing_index or missing_models or not (SAVED_MODELS / 'models.lock.json').is_file():
    raise FileNotFoundError(f'Thiếu Version 3 artifacts: index={missing_index}, models={missing_models}. Add Input {VERSION3_URL}')

MODELS.mkdir(parents=True, exist_ok=True)
for role in ['embedding', 'reranker', 'generator']:
    link, source = MODELS / role, SAVED_MODELS / role
    if not link.exists():
        link.symlink_to(source, target_is_directory=True)
shutil.copy2(SAVED_MODELS / 'models.lock.json', MODELS / 'models.lock.json')

shared_cfg = json.loads((CODE / 'config.json').read_text(encoding='utf-8'))
if not shared_cfg['training'].get('required') or not shared_cfg['generation'].get('load_in_4bit'):
    raise RuntimeError('Stage 1 bắt buộc QLoRA NF4 4-bit.')
if shared_cfg['training'].get('retrieval_mode') != 'lexical':
    raise RuntimeError('Training retrieval phải là lexical.')
if shared_cfg['evaluation'].get('primary_metric') != 'meteor' or shared_cfg['evaluation'].get('target_meteor') != 0.65:
    raise RuntimeError('Cấu hình phải ưu tiên METEOR và target 0.65.')
CFG.write_text(json.dumps(shared_cfg, ensure_ascii=False, indent=2), encoding='utf-8')

def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def run(*args):
    command = [sys.executable, '-m', 'legalqa', '--config', str(CFG), '--models', str(MODELS), *map(str, args)]
    print('Running:', ' '.join(command), flush=True)
    subprocess.run(command, cwd=CODE, check=True)

commit = subprocess.check_output(['git', '-C', str(CODE), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('Commit:', commit)
print('Evaluation objective:', shared_cfg['evaluation'])

## 3. Môi trường và kiểm định

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODE / 'requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=CODE, check=True)
freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(RUN_ROOT / 'environment.freeze.txt').write_text(freeze, encoding='utf-8')

## 4. Chuẩn bị dữ liệu và xác nhận full index

In [ ]:
run('prepare', '--train', TRAIN_PATH, '--test', TEST_PATH, '--output', DATA)
run('audit-models')
audit = json.loads((MODELS / 'parameter_audit.json').read_text(encoding='utf-8'))
if not audit['passes'] or audit['total_with_unmerged_lora'] >= 4_000_000_000:
    raise RuntimeError('Model vượt budget hoặc audit thất bại.')
index_manifest = json.loads((INDEX / 'index_manifest.json').read_text(encoding='utf-8'))
if index_manifest.get('chunks') != 407_107 or index_manifest.get('documents') != 8_507:
    raise RuntimeError(f'Không phải full index Version 3: {index_manifest}')
print('Reusing full index:', INDEX)
print(json.loads((DATA / 'data_report.json').read_text(encoding='utf-8')))

## 5. Tạo cache lexical cho 768 mẫu QLoRA

In [ ]:
SFT_TRAIN = DATA / 'train.sft.json'
SFT_QUESTIONS = DATA / 'train.sft.questions.json'
TRAIN_CACHE = RUN_ROOT / 'train.sft.lexical.retrieval.json'
run('prepare-sft', '--train', DATA / 'train.json', '--output', SFT_TRAIN)
sft_questions = json.loads(SFT_QUESTIONS.read_text(encoding='utf-8'))
expected = min(shared_cfg['training']['max_examples'], json.loads((DATA / 'data_report.json').read_text())['split_sizes']['train'])
if len(sft_questions) != expected:
    raise RuntimeError(f'Số mẫu SFT sai: {len(sft_questions)} != {expected}')
run('retrieve', '--questions', SFT_QUESTIONS, '--index', INDEX, '--output', TRAIN_CACHE, '--mode', 'lexical')
print('QLoRA examples:', len(sft_questions), '| retrieval mode: lexical')

## 6. Fine-tune và kiểm tra checkpoint resume

In [ ]:
args = ['fit', '--train', SFT_TRAIN, '--retrieval', TRAIN_CACHE, '--output', SFT_DIR, '--gpu', '0']
if RESUME_CHECKPOINT is not None:
    args += ['--resume', RESUME_CHECKPOINT]
if not (SFT_DIR / 'training_result.json').is_file():
    run(*args)

required_resume = {'adapter_config.json', 'adapter_model.safetensors', 'optimizer.pt', 'scheduler.pt', 'trainer_state.json', 'rng_state.pth'}
checkpoints = sorted(SFT_DIR.glob('checkpoint-*'), key=lambda p: int(p.name.split('-')[-1]))
if len(checkpoints) < 2:
    raise RuntimeError(f'Cần ít nhất 2 epoch checkpoints, hiện có: {[p.name for p in checkpoints]}')
for checkpoint in checkpoints:
    present = {p.name for p in checkpoint.iterdir() if p.is_file()}
    missing = sorted(required_resume - present)
    if missing:
        raise RuntimeError(f'{checkpoint.name} thiếu file resume: {missing}')
if not (SFT_DIR / 'adapter_last' / 'adapter_model.safetensors').is_file():
    raise RuntimeError('Thiếu adapter_last/adapter_model.safetensors')
if not (SFT_DIR / 'training_result.json').is_file():
    raise RuntimeError('Thiếu training_result.json')
print('QLoRA result:', json.loads((SFT_DIR / 'training_result.json').read_text(encoding='utf-8')))
print('Complete checkpoints:', [p.name for p in checkpoints])

## 7. Ghi manifest bàn giao cho Stage 2

In [ ]:
stage1_manifest = {
    'stage': 1,
    'quality_version': 'v8',
    'code_commit': commit,
    'config_sha256': file_sha256(CFG),
    'index_manifest_sha256': file_sha256(INDEX / 'index_manifest.json'),
    'training_result_sha256': file_sha256(SFT_DIR / 'training_result.json'),
    'checkpoints': [p.name for p in checkpoints],
    'sft_root': 'sft',
    'next_notebook': 'legalqa_main_02_select_retrieve.ipynb',
}
(RUN_ROOT / 'stage1_manifest.json').write_text(json.dumps(stage1_manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(stage1_manifest, ensure_ascii=False, indent=2))
print('SUCCESS Stage 1. Hãy Save output này và Add Input vào Stage 2.')